In [0]:
from pyspark.sql import functions as F

print("Starting Gold layer processing...")

In [0]:
df_customers = spark.table("02_prod_silver.transformed.customers")
df_products = spark.table("02_prod_silver.transformed.products")
df_stores = spark.table("02_prod_silver.transformed.stores")
df_sales = spark.table("02_prod_silver.transformed.sales")
df_exchange = spark.table("02_prod_silver.transformed.exchange_rates")

print("Silver tables loaded")

In [0]:
print("Creating dim_customers...")

dim_customers = df_customers.select(
    "customer_key",
    "gender",
    "continent"
)

In [0]:
print("Creating dim_products...")

dim_products = df_products.select(
    "product_key",
    "category",
    "subcategory"
)

In [0]:
print("Creating dim_stores...")

dim_stores = df_stores.select(
    "store_key",
    "country"
)

In [0]:
print("Creating dim_date...")

dim_date = df_sales.select("order_date") \
    .withColumn("year", F.year("order_date")) \
    .withColumn("month", F.month("order_date")) \
    .withColumn("month_name", F.date_format("order_date", "MMMM"))

In [0]:
print("Creating fact_sales...")

df_sales_fx = df_sales.join(
    df_exchange,
    (df_sales.order_date == df_exchange.date) &
    (df_sales.currency_code == df_exchange.currency),
    "left"
)

df_sales_full = df_sales_fx.join(
    df_products.select("product_key", "unit_price_usd"),
    "product_key",
    "left"
)

fact_sales = df_sales_full \
    .withColumn("revenue_usd",
        F.col("quantity") * F.col("unit_price_usd")
    ) \
    .withColumn("delivery_days",
        F.datediff("delivery_date", "order_date")
    )

In [0]:
fact_sales = fact_sales.select(
    "order_number",
    "order_date",
    "customer_key",
    "store_key",
    "product_key",
    "quantity",
    "revenue_usd",
    "delivery_days"
)

In [0]:
print("Writing Gold tables...")

tables = {
    "dim_customers": dim_customers,
    "dim_products": dim_products,
    "dim_stores": dim_stores,
    "dim_date": dim_date,
    "fact_sales": fact_sales
}

for name, df in tables.items():
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"03_prod_gold.models.{name}")
    
    print(f"Table created: {name}")

print("Gold layer completed")